In [ ]:
import sys
import os
import re
from typing import Optional
# Thêm thư mục gốc vào path để nhận diện thư mục src
sys.path.append(os.path.abspath(os.path.join('..')))

from src.utils import clean_honorific_only, remap_role_vietstock

# Test case xóa "Ông/bà" trước tên

In [ ]:

def test_clean_honorific_only():
    print("--- Đang chạy Test Cases cho clean_honorific_only ---")
    
    # Nhóm 1: Trường hợp thông thường (Có khoảng trắng)
    assert clean_honorific_only("Ông Nguyễn Văn A") == "Nguyễn Văn A"
    assert clean_honorific_only("Bà Lê Thị B") == "Lê Thị B"
    
    # Nhóm 2: Trường hợp có dấu chấm (Ông., Bà.)
    assert clean_honorific_only("Ông. Nguyễn Văn A") == "Nguyễn Văn A"
    assert clean_honorific_only("Bà. Lê Thị B") == "Lê Thị B"
    
    # Nhóm 3: Trường hợp viết hoa, viết thường xen kẽ
    assert clean_honorific_only("ông Nguyễn Văn A") == "Nguyễn Văn A"
    assert clean_honorific_only("bà Lê Thị B") == "Lê Thị B"
    
    # Nhóm 4: Trường hợp danh xưng kép hoặc lặp (Regex xử lý nhờ dấu +)
    assert clean_honorific_only("Ông bà Nguyễn Văn A") == "Nguyễn Văn A"
    assert clean_honorific_only("Ông Ông Nguyễn Văn A") == "Nguyễn Văn A"
    
    # Nhóm 5: Tên chứa từ "Ông" hoặc "Bà" nhưng không phải ở đầu (Không được xóa)
    assert clean_honorific_only("Lý Ông Trọng") == "Lý Ông Trọng"
    assert clean_honorific_only("Đỗ Thị Bà") == "Đỗ Thị Bà"
    
    # Nhóm 6: Tên không có danh xưng (Giữ nguyên)
    assert clean_honorific_only("Trần Văn C") == "Trần Văn C"
    
    # Nhóm 7: Dữ liệu đặc biệt (None, Rỗng, Số)
    assert clean_honorific_only(None) is None
    assert clean_honorific_only("") == ""
    assert clean_honorific_only(123) == 123  # Trả về nguyên bản nếu không phải string
    
    
    print("✅ TẤT CẢ TEST CASES ĐÃ VƯỢT QUA!")

# Chạy hàm test
test_clean_honorific_only()

--- Đang chạy Test Cases cho clean_honorific_only ---
✅ TẤT CẢ TEST CASES ĐÃ VƯỢT QUA!


# Test case mapping tên role của vietstock

In [130]:
import sys
import os
import re
from typing import Optional

# Cấu hình đường dẫn để import từ src
sys.path.append(os.path.abspath(os.path.join('..')))
# từ src.utils import remap_role_vietstock 

def test_remap_role_vietstock():
    print("--- Đang chạy Test Cases cho remap_role_vietstock ---")
    
    # 1. Kiểm tra ánh xạ đơn (Single mapping)
    assert remap_role_vietstock("CTHĐQT") == "Chủ tịch HĐQT"
    assert remap_role_vietstock("TGĐ") == "Tổng Giám đốc"
    assert remap_role_vietstock("KTT") == "Kế toán trưởng"
    print("✅ Nhóm 1: Ánh xạ đơn thành công.")

    # 2. Kiểm tra chức danh ghép với dấu "/" (Compound roles)
    # Lưu ý: Code của bạn dùng separator " / " (có khoảng trắng)
    assert remap_role_vietstock("CTHĐQT/TGĐ") == "Chủ tịch HĐQT / Tổng Giám đốc"
    print("✅ Nhóm 2: Chức danh ghép thành công.")

    # 3. Kiểm tra tính chính xác của Regex (Tránh lỗi thay thế substring)
    # "TV" không được đè lên "TVHĐQT"
    assert remap_role_vietstock("TVHĐQT") == "Thành viên HĐQT"
    assert remap_role_vietstock("TV") == "Thành viên"
    # "GĐ" không được đè lên "TGĐ" (nếu có regex \b)
    assert remap_role_vietstock("TGĐ") == "Tổng Giám đốc"
    print("✅ Nhóm 3: Regex Word Boundary (\b) hoạt động đúng (không lỗi substring).")

    # 4. Kiểm tra thứ tự ưu tiên (Longest match first)
    # "Phó CTHĐQT" phải được thay thế hoàn toàn thay vì chỉ thay "CTHĐQT"
    assert remap_role_vietstock("Phó CTHĐQT") == "Phó Chủ tịch HĐQT"
    print("✅ Nhóm 4: Thứ tự ưu tiên (dài nhất trước) thành công.")

    # 5. Kiểm tra các trường hợp biên (Edge cases)
    assert remap_role_vietstock(None) == "Khác"
    assert remap_role_vietstock("") == "Khác"
    assert remap_role_vietstock("***") == "Khác"
    print("✅ Nhóm 5: Các trường hợp rỗng/None thành công.")

    # 6. Kiểm tra từ không có trong từ điển (Unmapped words)
    # Những từ không có trong map phải được giữ nguyên
    assert remap_role_vietstock("Bảo vệ") == "Bảo vệ"
    assert remap_role_vietstock("TVHĐQT/Chuyên gia") == "Thành viên HĐQT / Chuyên gia"
    print("✅ Nhóm 6: Giữ nguyên từ không có trong map thành công.")

    # 7. Kiểm tra khoảng trắng trong input
    assert remap_role_vietstock(" CTHĐQT / TGĐ ") == "Chủ tịch HĐQT / Tổng Giám đốc"
    print("✅ Nhóm 7: Xử lý khoảng trắng dư thừa thành công.")

    print("\n🎉 CHÚC MỪNG: Tất cả test cases cho remap_role_vietstock đã PASSED!")

# Chạy test
test_remap_role_vietstock()

--- Đang chạy Test Cases cho remap_role_vietstock ---
✅ Nhóm 1: Ánh xạ đơn thành công.
✅ Nhóm 2: Chức danh ghép thành công.
✅ Nhóm 3: Regex Word Boundary ) hoạt động đúng (không lỗi substring).
✅ Nhóm 4: Thứ tự ưu tiên (dài nhất trước) thành công.
✅ Nhóm 5: Các trường hợp rỗng/None thành công.
✅ Nhóm 6: Giữ nguyên từ không có trong map thành công.
✅ Nhóm 7: Xử lý khoảng trắng dư thừa thành công.

🎉 CHÚC MỪNG: Tất cả test cases cho remap_role_vietstock đã PASSED!
